# 👔 Outfit Inspiration MCP Server

Your personal AI stylist that creates outfit recommendations and emails them to you in a beautiful, magazine-style format.

### How it works:
1. **Cell 1**: Import everything we need
2. **Cell 2**: Setup the MCP server + services (Reddit, Email, Style)
3. **Cell 3**: Define 3 essential tools
4. **Cell 4**: Define the resource (style interests file)
5. **Cell 5**: Define the AI prompt (creates natural, conversational outfit advice)
6. **Cell 6**: Run the server!

### What you'll get:
✨ **iPhone Messages Interface** - Emails that look like text messages from your stylish friend
- Season-specific background colors (Fall = peachy terracotta, Winter = icy blue, Spring = sage green, Summer = warm cream)
- Transparent white chat bubbles with glass effect (75% opacity)
- SF Pro font (authentic iOS look)
- 4 distinct sections: The Look, The Details, Where to Shop, Switch It Up
- Color-coded headings matching the season

🎨 **3 Essential Tools:**
1. `get_season_info()` - Current season & keywords
2. `create_personalized_outfit()` - AI creates outfits based on your style
3. `email_outfit_recommendation()` - Sends beautiful emails

### Email Setup:
Create a `.env` file in the project root with:
```
MCP_SMTP_FROM_EMAIL=your-email@gmail.com
MCP_SMTP_PASSWORD=your-gmail-app-password
```

Get Gmail App Password: https://myaccount.google.com/apppasswords

In [1]:
# CELL 1: Setup and Imports
# This cell loads all the dependencies we need

import os
import sys
from pathlib import Path
from datetime import datetime
from typing import List

import nest_asyncio
from fastmcp import FastMCP, Context
from dotenv import load_dotenv

# Load environment variables from project root
env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)

# Add server directory to Python path so we can import our services
sys.path.insert(0, str(Path.cwd().parent / "src" / "server"))

# Allow nested async calls (needed for Jupyter notebooks)
nest_asyncio.apply()

# Import our custom services
from services.email_service import EmailService
from services.reddit_service import RedditService
from services.season_service import get_current_season, get_season_keywords
from services.style_interests_service import StyleInterestsService
from config.settings import get_settings

print("✅ All imports successful!")
print(f"📁 Working directory: {Path.cwd()}")


✅ All imports successful!
📁 Working directory: /Users/tiffanylin/daily-outfit-agent/notebooks


In [2]:
# CELL 2: Initialize MCP Server and Services
# This creates our MCP server and initializes the services it will use

# Create the MCP server
mcp = FastMCP(
    name="outfit-inspiration-agent",
    instructions="""A simple outfit inspiration agent that helps you discover fashion ideas from Reddit.

CAPABILITIES:
- Fetch outfit posts from fashion subreddits
- Understand your style preferences from style_interests.md
- Provide season-appropriate recommendations
- Curate inspiration from multiple subreddits
- Send outfit inspiration to your email

Use this to get daily outfit inspiration based on the current season and your personal style!"""
)

# Initialize settings
settings = get_settings()
settings.data_dir.mkdir(parents=True, exist_ok=True)

# Initialize services
reddit_service = RedditService()
style_service = StyleInterestsService(
    settings.data_dir / "style_interests.md"
)

# Initialize email service (for sending outfit inspiration)
email_service = EmailService(
    {
        "server": "smtp.gmail.com",
        "port": 465,
        "use_tls": False,
        "use_ssl": True,
        "username": os.getenv("MCP_SMTP_FROM_EMAIL", ""),
        "password": os.getenv("MCP_SMTP_PASSWORD", ""),
        "from_email": os.getenv("MCP_SMTP_FROM_EMAIL", ""),
        "from_name": "Outfit Inspiration Agent",
    }
)

print("✅ MCP Server created: 'outfit-inspiration-agent'")
print("✅ Services initialized!")
print(f"📂 Style interests file: {settings.data_dir / 'style_interests.md'}")
print(f"📧 Email service ready (from: {os.getenv('MCP_SMTP_FROM_EMAIL', 'Not configured')})")


✅ MCP Server created: 'outfit-inspiration-agent'
✅ Services initialized!
📂 Style interests file: /Users/tiffanylin/daily-outfit-agent/src/server/data/style_interests.md
📧 Email service ready (from: lintoutfits@gmail.com)


In [3]:
# CELL 3: Define MCP Tools
# These are the functions that the client can call

@mcp.tool()
async def get_season_info() -> str:
    """
    Get current season and relevant fashion keywords.
    
    Returns:
        Current season and style keywords for that season
    """
    season = get_current_season()
    keywords = get_season_keywords(season)
    
    return f"""🌸 Current Season: {season.upper()}

Style Keywords for {season}:
{chr(10).join(f'- {kw}' for kw in keywords)}

Consider these elements when looking for outfit inspiration!"""


@mcp.tool()
async def create_personalized_outfit(
    occasion: str = "casual",
    limit_per_subreddit: int = 5
) -> str:
    """
    Create a personalized outfit recommendation based on Reddit inspiration and your style profile.
    This analyzes Reddit posts and creates specific outfit suggestions tailored to YOU.
    
    Args:
        occasion: What you're dressing for (casual, work, date, weekend, etc.)
        limit_per_subreddit: Number of inspiration posts to analyze
    
    Returns:
        Personalized outfit recommendation with specific pieces
    """
    # Get user's style profile
    interests = style_service.read_interests()
    
    # Get favorite subreddits
    favorite_subs = style_service.get_subreddits()
    if not favorite_subs:
        # Default based on gender
        gender = interests.get("gender", "").lower()
        if "female" in gender or "woman" in gender:
            favorite_subs = ["femalefashionadvice", "womensstreetwear", "outfits"]
        else:
            favorite_subs = ["malefashionadvice", "streetwear", "outfits"]
    
    # Fetch Reddit posts for inspiration
    all_posts = await reddit_service.fetch_multiple_subreddits(
        favorite_subs[:3],
        limit_per_subreddit
    )
    
    # Get current season
    season = get_current_season()
    season_keywords = get_season_keywords(season)
    
    # Build context for outfit creation
    context = f"""# Create a Personalized Outfit Recommendation
    
## User's Style Profile:
- Gender: {interests.get('gender', 'Not specified')}
- Style Preferences: {', '.join(interests.get('style_preferences', []))}
- Favorite Colors: {', '.join(interests.get('favorite_colors', []))}
- Body/Fit Preferences: {', '.join(interests.get('body_preferences', []))}
- Occasion: {occasion}
- Season: {season} (keywords: {', '.join(season_keywords)})

## Reddit Inspiration Posts:
"""
    # Add top posts as inspiration
    post_count = 0
    for subreddit, posts in all_posts.items():
        if posts and post_count < 10:  # Limit to 10 posts for context
            context += f"\n### From r/{subreddit}:\n"
            for post in posts[:3]:
                context += f"- {post['title']} ({post['score']} upvotes)\n"
                post_count += 1
    
    context += f"""

## Your Task:
Create a SPECIFIC outfit recommendation with actual clothing pieces. Include:

1. **Complete Outfit** - List 4-6 specific clothing items (e.g., "Black oversized turtleneck", not just "top")
2. **Why It Works** - Explain how this fits the user's style profile
3. **Styling Tips** - 2-3 tips on how to wear it
4. **Where to Find Similar** - Suggest 2-3 stores/brands that match this aesthetic
5. **Alternative Pieces** - Suggest 1-2 swaps for different vibes

Be specific! Use the user's color preferences, style preferences, and body preferences. Make it wearable and practical for {season}.
"""
    
    return context


@mcp.tool()
async def email_outfit_recommendation(
    outfit_recommendation: str,
    subject: str = ""
) -> str:
    """
    Send a personalized outfit recommendation to email.
    Use this to email the AI-generated outfit (not Reddit threads).
    
    Args:
        outfit_recommendation: The complete outfit recommendation text created by the AI
        subject: Custom email subject (optional)
    
    Returns:
        Confirmation message
    """
    # Get current date and season
    season = get_current_season()
    date_str = datetime.now().strftime("%B %d, %Y")
    
    # Use email service to create and send the outfit email
    result = email_service.send_outfit_email(
        outfit_recommendation=outfit_recommendation,
        season=season,
        date_str=date_str,
        subject=subject
    )
    
    if result["success"]:
        return f"""✅ Your personalized outfit has been emailed!

            📧 Sent to: {email_service.from_email}
            🌸 Season: {season.title()}
            📅 Date: {date_str}

            Check your inbox for your custom outfit recommendation! 💌"""
    else:
        return f"❌ Failed to send email: {result.get('error', 'Unknown error')}"


print("✅ Defined 3 tools:")
print("   1. get_season_info - Get current season & keywords")
print("   2. create_personalized_outfit - 🎨 Get context to CREATE outfit recommendations!")
print("   3. email_outfit_recommendation - 📧 Email YOUR AI-generated outfit!")


✅ Defined 3 tools:
   1. get_season_info - Get current season & keywords
   2. create_personalized_outfit - 🎨 Get context to CREATE outfit recommendations!
   3. email_outfit_recommendation - 📧 Email YOUR AI-generated outfit!


In [4]:
# CELL 4: Define MCP Resource
# This exposes your style preferences file to the client

@mcp.resource("file://style_interests.md")
async def get_style_interests() -> str:
    """
    User's style preferences and interests.
    Read this to understand what styles the user likes.
    """
    interests = style_service.read_interests()
    
    content = "# Your Style Interests\n\n"
    
    if interests.get("style_preferences"):
        content += "## Style Preferences\n"
        for pref in interests["style_preferences"]:
            content += f"- {pref}\n"
        content += "\n"
    
    if interests.get("favorite_colors"):
        content += "## Favorite Colors\n"
        for color in interests["favorite_colors"]:
            content += f"- {color}\n"
        content += "\n"
    
    if interests.get("subreddits"):
        content += "## Favorite Subreddits\n"
        for sub in interests["subreddits"]:
            content += f"- {sub}\n"
        content += "\n"
    
    if interests.get("occasions"):
        content += "## Occasions\n"
        for occasion in interests["occasions"]:
            content += f"- {occasion}\n"
        content += "\n"
    
    if interests.get("notes"):
        content += "## Notes\n"
        for note in interests["notes"]:
            content += f"- {note}\n"
    
    return content


print("✅ Defined resource: file://style_interests.md")


✅ Defined resource: file://style_interests.md


In [5]:
# CELL 5: Define MCP Prompt
# This gives the AI client a template for creating outfit boards

@mcp.prompt()
async def daily_outfit_inspiration() -> str:
    """Get personalized outfit recommendations based on your style profile and current season."""
    return """Create a personalized outfit recommendation for me today!

WORKFLOW:
1. First, call get_season_info() to understand what season it is
2. Then call create_personalized_outfit(occasion="casual", limit_per_subreddit=5) to get my style profile and Reddit inspiration
3. Analyze the context and write like a stylish friend texting outfit advice - be conversational and natural!

WRITING STYLE - This is CRUCIAL:
- Write like you're texting a friend, not writing a report
- NO bullet points, NO numbered lists, NO headers like "Complete Outfit:" or "Why it works:"
- Use casual language: "I'm thinking...", "you could wear...", "maybe try...", "would look so good"
- Write in natural flowing paragraphs
- Be enthusiastic but natural - use "!" sparingly
- Keep it personal and warm

STRUCTURE (use paragraph breaks for readability):
Break your response into 4 paragraphs separated by blank lines. DO NOT add headings - they're added automatically!

Paragraph 1 (becomes "The Look"): Naturally describe 4-6 specific pieces
"Okay so I'm thinking for today... a cream chunky knit sweater with high-waisted black straight-leg jeans. Throw on your white leather sneakers and maybe layer a long camel coat over everything if it's cold out."

Paragraph 2 (becomes "The Details"): Why it works + styling tips
"The whole vibe is so you - minimal and effortless but still put together. You could tuck the front of the sweater loosely for shape, or leave it untucked for more casual. Love how the cream and black is classic but the oversized fit keeps it modern."

Paragraph 3 (becomes "Where to Shop"): Shopping suggestions only
"You can find pieces like this at Everlane for basics, COS for modern cuts, or Mango for affordable options. Check out their knitwear and denim sections."

Paragraph 4 (becomes "Switch It Up"): Alternatives and variations
"If you want to switch it up, swap the jeans for olive trousers and it's more earthy, or go with black trousers for a dressier look. You could also try a black turtleneck instead of cream for a sleeker vibe."

NOTE: Headings are automatically added - just write the content!

WHAT TO INCLUDE:
- 4-6 specific clothing items (not generic - actual pieces like "a cream chunky knit sweater")
- Why it matches their style/colors/preferences (naturally woven in)
- 2-3 styling suggestions (casual mentions)
- Where to shop (1-2 stores, casually mentioned)
- Maybe one alternative idea

Remember: conversational paragraphs with breaks, NOT lists or headers!

When you're done, email it to me."""


print("✅ Defined prompt: daily_outfit_inspiration")


✅ Defined prompt: daily_outfit_inspiration


In [6]:
# CELL 6: Run the MCP Server!
# This starts the server on port 8081 (using streamable-http transport)
# Keep this cell running while you use the client in another terminal

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("🚀 Starting Outfit Inspiration MCP Server...")
print("📍 Server will run on: http://127.0.0.1:8081/mcp")
print("📡 Transport: streamable-http")
print("")
print("💡 To test the server:")
print("   1. Keep this cell running")
print("   2. Open a new terminal")
print("   3. cd client/")
print("   4. python outfit_client.py")
print("")
print("⏸️  Press the stop button (■) to stop the server")
print("="*60)

# Kill any existing process on port 8081
os.system("lsof -ti:8081 | xargs kill -9 2>/dev/null")

# Run the server
await mcp.run_async(transport="streamable-http", port=8081)


🚀 Starting Outfit Inspiration MCP Server...
📍 Server will run on: http://127.0.0.1:8081/mcp
📡 Transport: streamable-http

💡 To test the server:
   1. Keep this cell running
   2. Open a new terminal
   3. cd client/
   4. python outfit_client.py

⏸️  Press the stop button (■) to stop the server


╭────────────────────────────────────────────────────────────────────────────╮
│                                                                            │
│        _ __ ___  _____           __  __  _____________    ____    ____     │
│       _ __ ___ .'____/___ ______/ /_/  |/  / ____/ __ \  |___ \  / __ \    │
│      _ __ ___ / /_  / __ `/ ___/ __/ /|_/ / /   / /_/ /  ___/ / / / / /    │
│     _ __ ___ / __/ / /_/ (__  ) /_/ /  / / /___/ ____/  /  __/_/ /_/ /     │
│    _ __ ___ /_/    \____/____/\__/_/  /_/\____/_/      /_____(*)____/      │
│                                                                            │
│                                                                            │
│                                FastMCP  2.0                                │
│                                                                            │
│                                                                            │
│               🖥️  Server name:     outfit-inspiration-agent                 │
│               📦 Transport:       Streamable-HTTP                          │
│               🔗 Server URL:      http://127.0.0.1:8081/mcp                │
│                                                                            │
│               🏎️  FastMCP version: 2.12.4                                   │
│               🤝 MCP SDK version: 1.16.0                                   │
│                                                                            │
│               📚 Docs:            https://gofastmcp.com                    │
│               🚀 Deploy:          https://fastmcp.cloud                    │
│                                                                            │
╰────────────────────────────────────────────────────────────────────────────╯

[10/21/25 23:06:46] INFO     Starting MCP server 'outfit-inspiration-agent' with transport           ]8;id=236781;file:///Users/tiffanylin/agentic-ai-workshop-2025/notebooks/.venv/lib/python3.13/site-packages/fastmcp/server/server.py\server.py]8;;\:]8;id=360185;file:///Users/tiffanylin/agentic-ai-workshop-2025/notebooks/.venv/lib/python3.13/site-packages/fastmcp/server/server.py#1579\1579]8;;\
                             'streamable-http' on http://127.0.0.1:8081/mcp                                        

INFO:     Started server process [92254]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8081 (Press CTRL+C to quit)


INFO:     127.0.0.1:64727 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64729 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:64731 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64733 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64735 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64737 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64740 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64743 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:64751 - "POST /mcp HTTP/1.1" 200 OK


[10/21/25 23:07:11] INFO     Email sent successfully                                           ]8;id=656783;file:///Users/tiffanylin/daily-outfit-agent/src/server/services/email_service.py\email_service.py]8;;\:]8;id=933000;file:///Users/tiffanylin/daily-outfit-agent/src/server/services/email_service.py#300\300]8;;\

INFO:     127.0.0.1:64769 - "DELETE /mcp HTTP/1.1" 200 OK


INFO:     Shutting down
ERROR:    Cancel 0 running task(s), timeout graceful shutdown exceeded
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [92254]
